In [3]:
import os

# Nombre del archivo de salida
salida = "context_backend.txt"

# Abrimos el archivo de salida en modo escritura
with open(salida, "w", encoding="utf-8") as out:
    # Recorremos recursivamente el directorio actual
    for root, dirs, files in os.walk("./backend"):
        for file in files:
            if file.endswith(".py") or file.endswith(".txt") or file.endswith('.yaml') or file.endswith('Dockerfile') or file.endswith('.txt'):
                ruta_completa = os.path.join(root, file)
                try:
                    with open(ruta_completa, "r", encoding="utf-8") as f:
                        contenido = f.read()
                except Exception as e:
                    contenido = f"[ERROR AL LEER EL ARCHIVO: {e}]"

                # Escribir en el txt
                out.write(f"### Archivo: {ruta_completa}\n")
                out.write(contenido)
                out.write("\n---\n\n")

print(f"✅ Proceso completado. Los archivos .py fueron guardados en '{salida}'.")

✅ Proceso completado. Los archivos .py fueron guardados en 'context_backend.txt'.


In [4]:
import os

# Nombre del archivo de salida
salida = "context_frontend.txt"

# Abrimos el archivo de salida en modo escritura
with open(salida, "w", encoding="utf-8") as out:
    # Recorremos recursivamente el directorio actual
    for root, dirs, files in os.walk("./frontend"):
        # Evitar entrar en la carpeta node_modules si existe
        if "node_modules" in dirs:
            dirs.remove("node_modules")
            
        for file in files:
            if file.endswith(".tsx") or file.endswith(".ts") or file.endswith(".js") or file.endswith(".conf") or file.endswith("Dockerfile"):
                ruta_completa = os.path.join(root, file)
                try:
                    with open(ruta_completa, "r", encoding="utf-8") as f:
                        contenido = f.read()
                except Exception as e:
                    contenido = f"[ERROR AL LEER EL ARCHIVO: {e}]"

                # Escribir en el txt
                out.write(f"### Archivo: {ruta_completa}\n")
                out.write(contenido)
                out.write("\n---\n\n")

print(f"✅ Proceso completado. Los archivos .tsx y .ts fueron guardados en '{salida}'.")

✅ Proceso completado. Los archivos .tsx y .ts fueron guardados en 'context_frontend.txt'.


In [3]:
import os

# Nombre del archivo de salida
salida = "context_latentsync.txt"

# Abrimos el archivo de salida en modo escritura
with open(salida, "w", encoding="utf-8") as out:
    # Recorremos recursivamente el directorio actual
    for root, dirs, files in os.walk("./latentsync_service"):
        for file in files:
            if file.endswith(".py") or file.endswith("README.md") or file.endswith('.yaml') or file.endswith('Dockerfile') or file.endswith('.txt') or file.endswith('.sh'):
                ruta_completa = os.path.join(root, file)
                try:
                    with open(ruta_completa, "r", encoding="utf-8") as f:
                        contenido = f.read()
                except Exception as e:
                    contenido = f"[ERROR AL LEER EL ARCHIVO: {e}]"

                # Escribir en el txt
                out.write(f"### Archivo: {ruta_completa}\n")
                out.write(contenido)
                out.write("\n---\n\n")

print(f"✅ Proceso completado. Los archivos .py fueron guardados en '{salida}'.")

✅ Proceso completado. Los archivos .py fueron guardados en 'context_latentsync.txt'.


In [1]:
import os

# Nombre del archivo de salida
salida = "context_mcp_server.txt"

# Abrimos el archivo de salida en modo escritura
with open(salida, "w", encoding="utf-8") as out:
    # Recorremos recursivamente el directorio actual
    for root, dirs, files in os.walk("./mcp_server"):
        for file in files:
            if file.endswith(".py") or file.endswith("README.md") or file.endswith('.yaml') or file.endswith('Dockerfile') or file.endswith('.txt') or file.endswith('.sh'):
                ruta_completa = os.path.join(root, file)
                try:
                    with open(ruta_completa, "r", encoding="utf-8") as f:
                        contenido = f.read()
                except Exception as e:
                    contenido = f"[ERROR AL LEER EL ARCHIVO: {e}]"

                # Escribir en el txt
                out.write(f"### Archivo: {ruta_completa}\n")
                out.write(contenido)
                out.write("\n---\n\n")

print(f"✅ Proceso completado. Los archivos .py fueron guardados en '{salida}'.")

✅ Proceso completado. Los archivos .py fueron guardados en 'context_mcp_server.txt'.


In [16]:
import requests
from bs4 import BeautifulSoup
from urllib.parse import urljoin, urldefrag
import time

# ---- CONFIGURACIÓN ----

BASE_PREFIX = "https://google.github.io/adk-docs/"
EXCLUDE_PREFIXES = [
    "https://google.github.io/adk-docs/api-reference/python/",
    "https://google.github.io/adk-docs/api-reference/java/",
    "https://google.github.io/adk-docs/api-reference/",
    "https://google.github.io/adk-docs/contributing-guide/",
    "https://google.github.io/adk-docs/community/",
    "https://google.github.io/adk-docs/a2a/",
    "https://google.github.io/adk-docs/safety/",
    "https://google.github.io/adk-docs/grounding/vertex_ai_search_grounding/",
    "https://google.github.io/adk-docs/get-started/java/",
    "https://google.github.io/adk-docs/deploy/",
    "https://google.github.io/adk-docs/get-started/streaming/quickstart-streaming-java/",
]

START_URLS = [BASE_PREFIX]
OUTPUT_FILE = "google_adk_context.txt"


# ---- FUNCIONES ----

def normalize_url(url):
    """Elimina fragmentos (#...) y limpia espacios."""
    clean_url, _ = urldefrag(url)
    return clean_url.strip()


def is_valid_internal_link(url):
    """
    Acepta solo URLs dentro del dominio base y que NO empiecen por las rutas excluidas.
    Solo procesa .html o rutas terminadas en '/'.
    """
    if not url.startswith(BASE_PREFIX):
        return False
    if any(url.startswith(prefix) for prefix in EXCLUDE_PREFIXES):
        return False
    return url.endswith(".html") or url.endswith("/")


def get_links(url):
    """Extrae todos los enlaces internos válidos de una página."""
    try:
        resp = requests.get(url, timeout=10)
        resp.raise_for_status()
        soup = BeautifulSoup(resp.text, "html.parser")

        links = set()
        for a in soup.find_all("a", href=True):
            full_url = urljoin(url, a["href"])
            full_url = normalize_url(full_url)
            if is_valid_internal_link(full_url):
                links.add(full_url)
        return links

    except Exception as e:
        print(f"⚠️ Error al obtener enlaces de {url}: {e}")
        return set()


def get_text_from_url(url):
    """Extrae texto con formato básico y mantiene la indentación."""
    try:
        resp = requests.get(url, timeout=10)
        resp.raise_for_status()
        soup = BeautifulSoup(resp.text, "html.parser")

        # Elimina elementos no relevantes
        for tag in soup(["script", "style", "nav", "footer", "header", "aside"]):
            tag.decompose()

        # Preserva saltos de línea y sangrías para listas, bloques y código
        for br in soup.find_all("br"):
            br.replace_with("\n")
        for li in soup.find_all("li"):
            li.insert_before("\n- ")
        for pre in soup.find_all("pre"):
            pre.insert_before("\n")
            pre.insert_after("\n")

        # Extrae el texto respetando la estructura
        text = soup.get_text(separator="\n")

        # Limpia líneas vacías múltiples pero mantiene indentación
        cleaned = []
        for line in text.splitlines():
            if line.strip():
                cleaned.append(line.rstrip())
        return "\n".join(cleaned)

    except Exception as e:
        print(f"⚠️ Error al extraer texto de {url}: {e}")
        return ""


# ---- SCRAPING PRINCIPAL ----

visited = set()
to_visit = set(START_URLS)
all_texts = []

print("🚀 Iniciando scraping...")

while to_visit:
    url = to_visit.pop()
    if url in visited:
        continue

    print(f"📄 Visitando: {url}")
    visited.add(url)

    text = get_text_from_url(url)
    if text:
        all_texts.append(f"\n\n--- {url} ---\n\n{text}")

    new_links = get_links(url)
    for link in new_links:
        if link not in visited and is_valid_internal_link(link):
            to_visit.add(link)

    # Pequeño delay opcional para evitar saturar el servidor
    # time.sleep(0.2)

# ---- GUARDAR RESULTADO ----

with open(OUTPUT_FILE, "w", encoding="utf-8") as f:
    f.write("\n".join(all_texts))

print(f"\n✅ Scraping completado. Texto guardado en '{OUTPUT_FILE}'")


🚀 Iniciando scraping...
📄 Visitando: https://google.github.io/adk-docs/
📄 Visitando: https://google.github.io/adk-docs/callbacks/
📄 Visitando: https://google.github.io/adk-docs/agents/workflow-agents/
📄 Visitando: https://google.github.io/adk-docs/observability/phoenix/
📄 Visitando: https://google.github.io/adk-docs/get-started/
📄 Visitando: https://google.github.io/adk-docs/sessions/
📄 Visitando: https://google.github.io/adk-docs/tools/mcp-tools/
📄 Visitando: https://google.github.io/adk-docs/streaming/custom-streaming/
📄 Visitando: https://google.github.io/adk-docs/tools/built-in-tools/
📄 Visitando: https://google.github.io/adk-docs/sessions/session/
📄 Visitando: https://google.github.io/adk-docs/get-started/streaming/quickstart-streaming/
📄 Visitando: https://google.github.io/adk-docs/tools/third-party-tools/
📄 Visitando: https://google.github.io/adk-docs/get-started/python/
📄 Visitando: https://google.github.io/adk-docs/tools/
📄 Visitando: https://google.github.io/adk-docs/observab